# UD3.05 — Del DataFrame a la matriz de características

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**  
UD3 — NumPy y Pandas · 14 horas

Criterio 2.c · Material de partida de la práctica P3.2


## De qué va este cuaderno

Los cuatro cuadernos anteriores enseñan a manejar arrays y tablas. Este responde a otra
pregunta, y es la que evalúa el criterio 2.c: **dado un problema, qué tabla hay que
construir para poder resolverlo**.

Es un paso que se salta casi siempre, y es el que más decide el resultado. Un modelo
de aprendizaje automático no recibe un DataFrame: recibe dos cosas.

| Nombre | Qué es | Forma |
|---|---|---|
| `X` | La matriz de características. Una fila por observación, una columna por variable. Todo numérico. | `(n_observaciones, n_caracteristicas)` |
| `y` | El objetivo. Un valor por observación: lo que se quiere predecir. | `(n_observaciones,)` |

Todo lo que sigue es el trabajo de llegar de una tabla de ventas a esas dos cosas.
No se entrena nada aquí: entrenar es la UD5. Aquí se **define el modelo**, que es
distinto y anterior.

> **Nota**
>
> Este cuaderno no usa scikit-learn. Todo se hace con pandas y NumPy, a propósito:
> así se ve qué hace cada transformación por dentro antes de esconderla detrás de
> una llamada de biblioteca.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(20262027)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("numpy", np.__version__, "\u00b7 pandas", pd.__version__)


---

## 1. Las cuatro preguntas que hay que responder antes de escribir código

Definir el modelo es contestar cuatro preguntas por escrito. Si alguna queda sin
respuesta, el código que venga después no puede ser correcto, porque no se sabe qué
debería calcular.

1. **¿Cuál es la unidad de observación?** Una fila de `X` es *un* qué: un cliente, un
   pedido, un día, un producto en una tienda en un mes. La tabla de partida casi nunca
   está en la unidad que necesitas.
2. **¿Qué se quiere predecir, y de qué tipo es?** Un número (regresión), una etiqueta
   entre varias (clasificación), o ninguna cosa concreta porque solo se quieren agrupar
   observaciones parecidas (agrupamiento).
3. **¿Qué información estará disponible en el momento de predecir?** Esta es la que más
   se falla, y tiene nombre propio: **fuga de información**.
4. **¿Cómo se va a saber si funciona?** Qué se compara y contra qué.

Vamos con un caso concreto y las respondemos las cuatro.


### El caso

> Una tienda en línea quiere saber **qué clientes van a dejar de comprar**, para poder
> hacerles una oferta antes de que se vayan. Tiene el registro de todos los pedidos de
> los dos últimos años.

Las respuestas:

| Pregunta | Respuesta para este caso |
|---|---|
| Unidad de observación | **Un cliente.** Los datos vienen por pedido, así que habrá que agregar. |
| Qué se predice | **Etiqueta binaria:** ¿ha dejado de comprar? Clasificación. |
| Qué se sabe al predecir | Solo lo ocurrido **hasta la fecha de corte**. Nada posterior. |
| Cómo se mide | Proporción de clientes que se iban y el modelo detectó, y de falsas alarmas. |

La segunda y la tercera necesitan una decisión más: **hay que inventar el objetivo**.
En los datos no existe ninguna columna que diga "este cliente se fue". Hay que
definirla, y la definición es una decisión de negocio, no técnica.


In [ ]:
# Registro de pedidos: la tabla tal como sale del sistema de la tienda.
#
# Los datos son sinteticos, pero no son ruido: cada cliente tiene su propio
# ritmo de compra y su propia fecha de baja, y a partir de esa fecha deja de
# aparecer. Eso es lo que hace que el problema tenga solucion. Con clientes
# uniformemente aleatorios no habria nada que predecir, y el cuaderno
# enseñaria a construir una X para un modelo que no puede funcionar.
INICIO = pd.Timestamp("2023-01-01")
FIN = pd.Timestamp("2024-12-31")
n_clientes = 400

filas = []
for i in range(1, n_clientes + 1):
    # Cada cliente empieza en un momento distinto y compra a su propio ritmo.
    alta = INICIO + pd.Timedelta(days=int(rng.integers(0, 400)))
    dias_entre = float(rng.gamma(shape=4.0, scale=9.0)) + 7.0

    # Un 35 % de los clientes se da de baja en algun momento. El resto sigue
    # comprando hasta el final del periodo.
    if rng.random() < 0.35:
        baja = alta + pd.Timedelta(days=int(rng.integers(60, 600)))
    else:
        baja = FIN + pd.Timedelta(days=1)

    # Preferencias estables del cliente: la categoria y el canal no cambian
    # en cada pedido, y por eso son informativos al agregarlos.
    pesos_cat = rng.dirichlet([1.2, 1.2, 1.2])
    prob_movil = float(rng.beta(2, 2))
    gasto_tipico = float(rng.gamma(shape=3.0, scale=25.0)) + 15.0

    fecha = alta
    while fecha < min(baja, FIN):
        filas.append((
            f"C{i:04d}",
            fecha,
            round(rng.gamma(shape=4.0, scale=gasto_tipico / 4.0) + 5.0, 2),
            rng.choice(["informatica", "audio", "telefonia"], p=pesos_cat),
            "movil" if rng.random() < prob_movil else "web",
        ))
        fecha = fecha + pd.Timedelta(days=float(rng.exponential(dias_entre)))

pedidos = (pd.DataFrame(filas, columns=["cliente", "fecha", "importe",
                                        "categoria", "canal"])
           .sort_values("fecha").reset_index(drop=True))

print(f"{len(pedidos)} pedidos de {pedidos['cliente'].nunique()} clientes")
print(f"de {pedidos['fecha'].min().date()} a {pedidos['fecha'].max().date()}")
pedidos.head()


---

## 2. La fecha de corte, y por qué sin ella todo lo demás está mal

La tentación es calcular las características con toda la tabla y la etiqueta también.
Eso produce un modelo que parece buenísimo y que en producción no acierta nada, porque
para predecir usa información que en el momento de predecir todavía no existía. Es la
**fuga de información**, y es el error más caro de este oficio.

La forma de evitarlo es partir el tiempo en dos por una **fecha de corte**:

```
      observación                          |        resultado
  <---------------------------------------->|<----------------->
  de donde salen las CARACTERÍSTICAS (X)    |  de donde sale la ETIQUETA (y)
                                          corte
```

Todo lo que va en `X` se calcula **solo** con lo de la izquierda. La etiqueta se mira
**solo** a la derecha. Nunca se cruzan.


In [ ]:
CORTE = pd.Timestamp("2024-07-01")
VENTANA_RESULTADO = pd.Timedelta(days=180)

observacion = pedidos[pedidos["fecha"] < CORTE]
resultado = pedidos[(pedidos["fecha"] >= CORTE) &
                    (pedidos["fecha"] < CORTE + VENTANA_RESULTADO)]

print(f"Ventana de observacion: {observacion['fecha'].min().date()} "
      f"a {observacion['fecha'].max().date()}  ({len(observacion)} pedidos)")
print(f"Ventana de resultado:   {resultado['fecha'].min().date()} "
      f"a {resultado['fecha'].max().date()}  ({len(resultado)} pedidos)")


### La etiqueta

Definición: **un cliente ha dejado de comprar si no hizo ningún pedido en los 180 días
siguientes al corte**.

Los 180 días son una decisión, no un dato. Con 30 días se etiquetaría como perdido a
cualquiera que compre trimestralmente; con 500 no queda tiempo para reaccionar. El
número sale del negocio: cada cuánto compra un cliente normal. Lo que **no** vale es
no escribirlo: quien lea el análisis tiene que poder discutir esa cifra.

Ojo a un detalle: la población de la que se predice son los clientes que **estaban
activos antes del corte**. Un cliente que aparece por primera vez después del corte no
puede formar parte de `X`, porque en el momento de predecir no existía.


In [ ]:
clientes = pd.Index(observacion["cliente"].unique(), name="cliente").sort_values()
compro_despues = set(resultado["cliente"])

y = pd.Series([c not in compro_despues for c in clientes],
              index=clientes, name="ha_dejado_de_comprar").astype(int)

print(f"Clientes en la poblacion: {len(y)}")
print(f"Se han ido:  {y.sum()} ({y.mean() * 100:.1f} %)")
print(f"Se quedan:   {(1 - y).sum()} ({(1 - y).mean() * 100:.1f} %)")
print()
print("Este reparto importa: si una clase fuera el 2 % del total, un modelo que")
print("conteste siempre 'se queda' acertaria el 98 % de las veces y no serviria")
print("para nada. La metrica hay que elegirla mirando este numero.")


---

## 3. Construir `X`: de pedidos a clientes

La tabla está en pedidos y la necesitamos en clientes. Eso es exactamente el
`groupby` del cuaderno anterior, pero ahora con una intención: cada agregación que
escribas es **una hipótesis sobre qué predice el abandono**.

Las tres primeras son un clásico con nombre propio, **RFM**:

| Sigla | Qué mide | Cómo se calcula |
|---|---|---|
| **R**ecencia | Cuánto hace del último pedido | días entre el último pedido y el corte |
| **F**recuencia | Cuántas veces ha comprado | número de pedidos |
| **M**onetario | Cuánto ha gastado | suma de los importes |

Y se añaden las que el caso sugiere: si compra por móvil o por web, en cuántas
categorías, cuánto gasta de media, y cuánto tiempo lleva siendo cliente.


In [ ]:
g = observacion.groupby("cliente")

X = pd.DataFrame({
    # RFM
    "recencia_dias": (CORTE - g["fecha"].max()).dt.days,
    "frecuencia": g["fecha"].count(),
    "monetario": g["importe"].sum(),
    # Comportamiento
    "ticket_medio": g["importe"].mean(),
    "antiguedad_dias": (CORTE - g["fecha"].min()).dt.days,
    "categorias_distintas": g["categoria"].nunique(),
    "proporcion_movil": g["canal"].apply(lambda s: (s == "movil").mean()),
})

# Una caracteristica derivada: cada cuantos dias compra, de media. Dice mas que
# la frecuencia a secas, porque la normaliza por el tiempo que lleva siendo cliente.
X["dias_entre_pedidos"] = (X["antiguedad_dias"] / X["frecuencia"]).round(1)

X = X.reindex(clientes)
print(X.shape)
X.head()


In [ ]:
# Comprobacion obligatoria: X e y tienen que estar alineados fila a fila.
#
# Esto no es una formalidad. Si las dos tablas salen de dos groupby distintos y
# una tiene un cliente que la otra no, pandas los alinea por indice y aparecen
# NaN; si se convierten a NumPy sin comprobarlo, las filas se desplazan y cada
# cliente acaba con la etiqueta de otro. El modelo entrena con ruido y nadie lo ve.
assert X.index.equals(y.index), "X e y no estan alineados"
assert not X.isna().any().any(), f"quedan ausentes:\n{X.isna().sum()}"
print(f"X: {X.shape}   y: {y.shape}   alineados y sin ausentes")


---

## 4. Lo que un modelo no sabe leer

`X` ya es una tabla por cliente, pero todavía no es una matriz de características.
Faltan dos cosas.

### 4.1. Las variables categóricas

Un modelo trabaja con números. Una columna con los valores `web` y `movil` hay que
convertirla, y **cómo** se convierta cambia lo que el modelo puede aprender:

| Forma | Qué hace | Cuándo sirve |
|---|---|---|
| Codificación ordinal | `web`→0, `movil`→1 | Solo si el orden significa algo (bajo < medio < alto) |
| Indicadores (*one-hot*) | Una columna 0/1 por valor | El caso normal: no inventa ningún orden |

El error habitual es usar la ordinal con categorías que no tienen orden. Si a
`informatica`, `audio` y `telefonia` se les asigna 0, 1 y 2, se le está diciendo al
modelo que audio está *entre* informática y telefonía, lo cual no quiere decir nada.

En pandas, los indicadores se generan con `pd.get_dummies`.


In [ ]:
# Categoria preferida de cada cliente: la que mas veces ha comprado.
preferida = (observacion.groupby(["cliente", "categoria"])["importe"]
             .sum().unstack(fill_value=0).idxmax(axis=1)
             .rename("categoria_preferida").reindex(clientes))

print(preferida.value_counts().to_string())
print()

# drop_first=True quita una de las columnas indicadoras. No se pierde informacion:
# si no es audio ni telefonia, es informatica. Se hace porque las tres columnas
# juntas son redundantes, y esa redundancia estorba a los modelos lineales.
indicadores = pd.get_dummies(preferida, prefix="cat", drop_first=True).astype(int)
X = X.join(indicadores)

print(X.dtypes.to_string())


### 4.2. Las escalas

Mira los órdenes de magnitud de las columnas de `X`. `monetario` va en cientos de
euros y `categorias_distintas` vale 1, 2 o 3. Para los modelos que miden distancias
entre observaciones (vecinos más cercanos, k-medias) o que ajustan pesos por descenso
de gradiente (regresión logística, redes neuronales), esa diferencia de escala hace
que la columna grande domine y las pequeñas no cuenten.

La corrección se llama **normalización** y ya la viste en el cuaderno UD3.01. Lo que
importa aquí es *cuándo* se aplica:

> **La media y la desviación se calculan solo con los datos de entrenamiento**
>
> Si se normaliza con la media de la tabla completa y luego se parte en entrenamiento
> y prueba, la media de la parte de prueba ha entrado en el cálculo. Es fuga de
> información otra vez, más disimulada. En la UD5 esto se resuelve con las
> tuberías (*pipelines*) de scikit-learn; el motivo es este.

In [ ]:
caracteristicas = X.columns.tolist()

print("Escalas de las caracteristicas:")
print(X[caracteristicas].agg(["min", "max", "mean", "std"]).T.round(2).to_string())


In [ ]:
# Particion temporal frente a particion aleatoria.
#
# Cuando el problema tiene tiempo dentro, partir al azar es hacer trampa: se
# entrena con clientes de julio para predecir clientes de marzo, y eso en
# produccion no pasa nunca. Aqui se parte por antiguedad: se entrena con los
# clientes mas viejos y se prueba con los mas recientes.
orden = X["antiguedad_dias"].sort_values(ascending=False).index
n_entrena = int(len(orden) * 0.75)
idx_entrena, idx_prueba = orden[:n_entrena], orden[n_entrena:]

X_entrena, X_prueba = X.loc[idx_entrena], X.loc[idx_prueba]
y_entrena, y_prueba = y.loc[idx_entrena], y.loc[idx_prueba]

# La media y la desviacion, SOLO del entrenamiento.
media = X_entrena.mean()
desv = X_entrena.std().replace(0, 1.0)

X_entrena_norm = (X_entrena - media) / desv
X_prueba_norm = (X_prueba - media) / desv     # se aplican las MISMAS

print(f"Entrenamiento: {X_entrena.shape}   se van el {y_entrena.mean() * 100:.1f} %")
print(f"Prueba:        {X_prueba.shape}   se van el {y_prueba.mean() * 100:.1f} %")
print()
print("Media de las caracteristicas normalizadas de entrenamiento (deberia ser ~0):")
print(X_entrena_norm.mean().round(6).to_string())
print()
print("Media de las de prueba (NO tiene por que ser 0, y eso es correcto):")
print(X_prueba_norm.mean().round(3).to_string())


---

## 5. Ya está: esto es lo que recibe un modelo

Cuatro objetos de NumPy, y la lista de nombres para poder interpretar después qué
columna era cada cosa. Ni una fecha, ni un texto, ni un ausente.


In [ ]:
X_entrena_np = X_entrena_norm.to_numpy(dtype=np.float64)
X_prueba_np = X_prueba_norm.to_numpy(dtype=np.float64)
y_entrena_np = y_entrena.to_numpy(dtype=np.int64)
y_prueba_np = y_prueba.to_numpy(dtype=np.int64)

print(f"X_entrena  {X_entrena_np.shape}  {X_entrena_np.dtype}")
print(f"y_entrena  {y_entrena_np.shape}  {y_entrena_np.dtype}")
print(f"X_prueba   {X_prueba_np.shape}  {X_prueba_np.dtype}")
print(f"y_prueba   {y_prueba_np.shape}  {y_prueba_np.dtype}")
print()
print(f"{len(caracteristicas)} caracteristicas:")
for i, nombre in enumerate(caracteristicas):
    print(f"  {i:2d}  {nombre}")
print()
assert np.isfinite(X_entrena_np).all() and np.isfinite(X_prueba_np).all()
print("Ni un NaN ni un infinito. Listo para la UD5.")


### El punto de referencia, antes de cualquier modelo

Antes de entrenar nada hay que saber **contra qué se compara**. La regla más tonta que
resuelva el problema es el punto de referencia, y más de un proyecto se ha cerrado al
descubrir que el modelo sofisticado no la superaba.

Aquí la regla tonta es evidente: *quien no ha comprado en mucho tiempo se va a ir*.
Una sola columna y un umbral.


In [ ]:
# Punto de referencia: predecir 'se va' si la recencia pasa de un umbral.
def evalua(prediccion, verdad):
    vp = int(((prediccion == 1) & (verdad == 1)).sum())
    fp = int(((prediccion == 1) & (verdad == 0)).sum())
    fn = int(((prediccion == 0) & (verdad == 1)).sum())
    precision = vp / (vp + fp) if vp + fp else 0.0
    exhaustividad = vp / (vp + fn) if vp + fn else 0.0
    f1 = (2 * precision * exhaustividad / (precision + exhaustividad)
          if precision + exhaustividad else 0.0)
    return precision, exhaustividad, f1

print(f"{'umbral':>8}  {'precision':>10}  {'exhaustiv.':>10}  {'F1':>6}")
print("-" * 40)
mejor = (0.0, None)
for umbral in (30, 60, 90, 120, 150, 180, 240):
    pred = (X_prueba["recencia_dias"] > umbral).astype(int).to_numpy()
    p, e, f1 = evalua(pred, y_prueba_np)
    print(f"{umbral:>8}  {p:>10.3f}  {e:>10.3f}  {f1:>6.3f}")
    if f1 > mejor[0]:
        mejor = (f1, umbral)

print()
print(f"Mejor punto de referencia: recencia > {mejor[1]} dias, F1 = {mejor[0]:.3f}")
print("Cualquier modelo de la UD5 tiene que superar esta cifra para justificarse.")


---

## 6. La ficha del modelo

Todo lo anterior se resume en un documento corto que **es lo que hay que entregar en la
P3.2**, antes de escribir una línea de análisis. Si no se puede rellenar, es que el
problema no está definido.

| Apartado | En este caso |
|---|---|
| Pregunta de negocio | ¿Qué clientes van a dejar de comprar, para poder retenerlos? |
| Unidad de observación | Un cliente activo antes del 1 de julio de 2024 |
| Tipo de tarea | Clasificación binaria |
| Objetivo (`y`) | 1 si no hay ningún pedido en los 180 días siguientes al corte |
| Por qué 180 días | Es algo más del doble del intervalo típico entre pedidos: por debajo se etiquetaría como perdido a quien compra por temporada |
| Características (`X`) | RFM, ticket medio, antigüedad, categorías distintas, proporción de móvil, días entre pedidos, categoría preferida en indicadores |
| Fuga de información | Evitada por fecha de corte: `X` solo mira antes, `y` solo después |
| Partición | Temporal por antigüedad, 75/25. No aleatoria: el problema tiene tiempo dentro |
| Métrica | F1 sobre la clase minoritaria. La exactitud no sirve con clases desequilibradas |
| Punto de referencia | Umbral sobre la recencia, una sola columna |
| Qué se hará con el resultado | Ofrecer descuento a los clientes marcados. Un falso positivo cuesta un descuento; un falso negativo, un cliente |

La última fila es la que decide la métrica, y casi nunca se escribe. Si un falso
negativo cuesta mucho más que un falso positivo, hay que preferir la exhaustividad a la
precisión, y eso se decide **antes** de entrenar, no después de mirar los resultados.


---

## Ejercicios

### Ejercicio 1. Mover el corte

Repite el análisis con `CORTE = pd.Timestamp("2024-01-01")` y ventana de resultado de
180 días. Responde: ¿cambia la proporción de clientes que se van? ¿Cambia el mejor
umbral del punto de referencia? ¿Qué te dice eso sobre la estabilidad del modelo?

### Ejercicio 2. Fabricar una fuga

Añade a `X` una característica calculada con la ventana de resultado: por ejemplo
`pedidos_despues_del_corte`. Vuelve a medir el punto de referencia usando esa columna.
Explica en una celda de texto por qué el resultado es magnífico y por qué es inútil.

### Ejercicio 3. Cambiar la unidad de observación

La misma tabla de pedidos, otra pregunta: **¿cuánto se va a vender la semana que
viene?** Escribe la ficha del modelo completa para esa pregunta. Cambian la unidad de
observación, el tipo de tarea, la métrica y la partición. No hace falta escribir el
código: hace falta que la ficha sea coherente.

### Ejercicio 4. Codificación equivocada

Sustituye los indicadores de `categoria_preferida` por una codificación ordinal
(`informatica`→0, `audio`→1, `telefonia`→2). Argumenta qué le estás diciendo al modelo
que no es verdad, y pon un ejemplo de variable categórica en la que la codificación
ordinal **sí** sería la correcta.

---

## Lo que hay que llevarse de aquí

1. Un modelo recibe `X` e `y`, no un DataFrame. Construir esas dos cosas es la mitad
   del trabajo.
2. La unidad de observación de la tabla de partida casi nunca es la que necesitas:
   `groupby` es la herramienta que las reconcilia.
3. **La fecha de corte no es opcional.** Sin ella hay fuga de información, y un modelo
   con fuga acierta en las pruebas y falla en producción.
4. El objetivo hay que definirlo, y la definición es una decisión discutible que se
   escribe y se justifica.
5. La métrica se elige mirando el reparto de clases y el coste de cada tipo de error,
   antes de entrenar.
6. Sin punto de referencia no se sabe si un modelo aporta algo.

Esto es lo que el criterio 2.c llama *definir el modelo que se quiere implementar según
el problema planteado*. Implementarlo es la UD5.
